# Count Green Highlights in Mann-Whitney Results

Scans every `.xlsx` file in a folder (the Mann-Whitney U test results, e.g.
`mannwhiteneyu_temperature.xlsx` / `mannwhiteneyu_co2.xlsx` / `mannwhiteneyu_drought.xlsx`),
and for both sheets (`C. arabica` and `C. canephora`) in each file, identifies every
green-highlighted cell (a significant p-value, 0 < p <= 0.05).

For each green cell we recover:
- the **metabolite name** (from column A of that row), and
- the **full contrast label** (reconstructed from the merged multi-level column header,
  e.g. `25/20°C vs 37/28°C | 380ppm | without`)

so that counts of "unique metabolites" and "unique contrasts" are based on actual
identities, not on row/column positions -- a metabolite highlighted in both the
temperature file and the drought file is one unique metabolite, not two.

Uses `openpyxl` directly (not `pandas.read_excel`, which discards formatting) so
actual cell fill colors can be read.

In [1]:
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook


## Config

In [2]:
RESULTS_FOLDER = r"C:\Users\papan\Desktop\Coffeeomics\metabolomics\mann_whitney_results"
SHEET_NAMES    = ["C. arabica", "C. canephora"]
GREEN_HEX      = "32A852"   # matches "background-color: #32a852"

OUTPUT_PER_FILE_CSV    = "green_highlights_per_file.csv"
OUTPUT_RECORDS_CSV     = "green_highlight_records.csv"
OUTPUT_PER_SPECIES_CSV = "green_highlights_per_species.csv"


## Color matching

In [3]:
def is_green(cell):
    """True if this cell's fill matches GREEN_HEX (solid fill)."""
    fill = cell.fill
    if fill is None or fill.patternType != "solid":
        return False

    color = fill.fgColor
    if color is None or color.type != "rgb" or not color.rgb:
        return False

    return color.rgb[-6:].upper() == GREEN_HEX.upper()


## Reconstruct row/column labels

`styled.to_excel(..., merge_cells=True)` writes the MultiIndex column header as
several stacked rows, merging repeated labels horizontally -- so `openpyxl` only
stores the value in the *top-left* cell of each merge and returns `None` for the
rest. To recover the real label for any given column we forward-fill each header
row left-to-right.

The bottom-most header row is always `u_statistic` / `p_value` (the innermost
level of the export). We detect it automatically -- that row number tells us
both where the header block ends and where the data starts.

In [4]:
def get_header_and_labels(ws):
    """
    Returns:
    - stat_row: 1-indexed row number containing the 'u_statistic'/'p_value' labels
      (the innermost, bottom-most header row)
    - col_labels: {col_idx: (level0, level1, ..., 'u_statistic'|'p_value')},
      forward-filled across merged header cells
    """
    max_col = ws.max_column
    max_row = ws.max_row

    stat_row = None
    for r in range(1, max_row + 1):
        row_values = [ws.cell(row=r, column=c).value for c in range(1, max_col + 1)]
        if "p_value" in row_values or "u_statistic" in row_values:
            stat_row = r
            break

    if stat_row is None:
        raise ValueError("Could not find header row containing 'u_statistic'/'p_value'")

    header_rows = []
    for r in range(1, stat_row + 1):
        values = []
        last = None
        for c in range(1, max_col + 1):
            v = ws.cell(row=r, column=c).value
            if v is not None:
                last = v
            values.append(last)
        header_rows.append(values)

    col_labels = {}
    for c in range(1, max_col + 1):
        levels = tuple(header_rows[lvl][c - 1] for lvl in range(stat_row))
        col_labels[c] = levels

    return stat_row, col_labels


## Per-sheet counting + record extraction

In [5]:
def count_sheet_highlights(ws):
    """
    Returns (n_highlighted_rows, total_highlights, n_highlighted_columns)
    for a single worksheet, based on raw grid position (kept for per-file detail).
    """
    highlighted_rows = set()
    highlighted_cols = set()
    total_highlights = 0

    for row in ws.iter_rows():
        for cell in row:
            if cell.value is None:
                continue
            if is_green(cell):
                total_highlights += 1
                highlighted_rows.add(cell.row)
                highlighted_cols.add(cell.column)

    return len(highlighted_rows), total_highlights, len(highlighted_cols)


def extract_highlighted_records(ws, file_name, species):
    """
    Returns a list of dicts, one per green cell:
    {file, species, metabolite, contrast}
    where 'metabolite' is the real row label (column A) and 'contrast' is the
    real reconstructed column label (all header levels except the trailing
    'p_value' marker, joined with " | ").
    """
    stat_row, col_labels = get_header_and_labels(ws)
    records = []

    for row in ws.iter_rows(min_row=stat_row + 1):
        for cell in row:
            if cell.value is None or not is_green(cell):
                continue

            levels = col_labels.get(cell.column, ())
            if not levels or levels[-1] != "p_value":
                # highlighting was only ever applied to p_value columns;
                # skip anything else as a safety net
                continue

            metabolite = ws.cell(row=cell.row, column=1).value
            contrast = " | ".join(str(x) for x in levels[:-1])

            records.append({
                "file": file_name,
                "species": species,
                "metabolite": metabolite,
                "contrast": contrast,
            })

    return records


## Scan every file in the folder

In [6]:
folder = Path(RESULTS_FOLDER)
xlsx_files = sorted(folder.glob("*.xlsx"))

if not xlsx_files:
    print(f"No .xlsx files found in: {folder}")

per_file_rows = []
all_records = []

for path in xlsx_files:
    print(f"Reading: {path.name}")
    wb = load_workbook(path, data_only=True)

    for sheet_name in SHEET_NAMES:
        if sheet_name not in wb.sheetnames:
            print(f"  Warning: sheet {sheet_name!r} not found in {path.name} "
                  f"(available: {wb.sheetnames}) -- skipped")
            continue

        ws = wb[sheet_name]

        n_rows, n_total, n_cols = count_sheet_highlights(ws)
        per_file_rows.append({
            "file": path.name,
            "species": sheet_name,
            "n_highlighted_rows": n_rows,
            "total_highlights": n_total,
            "n_highlighted_columns": n_cols,
        })
        print(f"  {sheet_name}: {n_rows} row(s), {n_total} total highlight(s), "
              f"{n_cols} column(s)")

        all_records.extend(extract_highlighted_records(ws, path.name, sheet_name))

    wb.close()

per_file_df = pd.DataFrame(per_file_rows)
records_df = pd.DataFrame(all_records)
per_file_df


Reading: mannwhiteneyu_co2.xlsx
  C. arabica: 18 row(s), 19 total highlight(s), 6 column(s)
  C. canephora: 14 row(s), 15 total highlight(s), 5 column(s)
Reading: mannwhiteneyu_drought.xlsx
  C. arabica: 43 row(s), 101 total highlight(s), 12 column(s)
  C. canephora: 35 row(s), 80 total highlight(s), 11 column(s)
Reading: mannwhiteneyu_temperature.xlsx
  C. arabica: 40 row(s), 235 total highlight(s), 23 column(s)
  C. canephora: 44 row(s), 201 total highlight(s), 23 column(s)


,file,species,n_highlighted_rows,total_highlights,n_highlighted_columns
0,mannwhiteneyu_co2.xlsx,C. arabica,18,19,6
1,mannwhiteneyu_co2.xlsx,C. canephora,14,15,5
2,mannwhiteneyu_drought.xlsx,C. arabica,43,101,12
3,mannwhiteneyu_drought.xlsx,C. canephora,35,80,11
4,mannwhiteneyu_temperature.xlsx,C. arabica,40,235,23
5,mannwhiteneyu_temperature.xlsx,C. canephora,44,201,23


## Per-species summary (unique metabolites & unique contrasts across all treatments)

This is the corrected version. Instead of summing per-file row/column *counts*
(which double-counts a metabolite highlighted in more than one file), we take
**unique values** of the actual metabolite name and actual contrast identity
across all three files combined.

- `n_unique_metabolites` -- number of distinct metabolites significantly
  differentially abundant in **at least one** contrast, across temperature,
  CO2, and drought comparisons combined.
- `n_unique_contrasts` -- number of distinct pairwise contrasts (columns) that
  had **at least one** significant metabolite, across all three files combined.
  Contrast identity includes the source file, so a "without vs medium" contrast
  in the drought file is never confused with a same-named contrast elsewhere.
- `total_highlights` -- total number of significant (metabolite, contrast) hits
  -- i.e. every green cell, unchanged from before.

In [7]:
records_df["contrast_id"] = records_df["file"] + " :: " + records_df["contrast"]

per_species_df = (
    records_df
    .groupby("species")
    .agg(
        n_unique_metabolites=("metabolite", "nunique"),
        n_unique_contrasts=("contrast_id", "nunique"),
        total_highlights=("metabolite", "size"),
    )
    .reset_index()
)
per_species_df


,species,n_unique_metabolites,n_unique_contrasts,total_highlights
0,C. arabica,46,41,355
1,C. canephora,48,39,296


## Save results

In [8]:
per_file_df.to_csv(OUTPUT_PER_FILE_CSV, index=False)
records_df.drop(columns="contrast_id").to_csv(OUTPUT_RECORDS_CSV, index=False)
per_species_df.to_csv(OUTPUT_PER_SPECIES_CSV, index=False)

print(f"Saved: {OUTPUT_PER_FILE_CSV}")
print(f"Saved: {OUTPUT_RECORDS_CSV}")
print(f"Saved: {OUTPUT_PER_SPECIES_CSV}")


Saved: green_highlights_per_file.csv
Saved: green_highlight_records.csv
Saved: green_highlights_per_species.csv
